 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison 
Name           : John Ryan         <br>
Student Number : L00007202         <br>
Due Date       : 12th May 2026     <br>
Assignment     : CA2               <br>
Module         : AI for Vision and NLP    <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level
An image of your working pipeline at high level can be inserted here



# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [14]:
# pip installs
# pip install -r requirements.txt

## Imports

In [15]:
# imports
import pandas as pd
import pymupdf
import pytesseract
import os
from PIL import Image
import cv2
import numpy as np
import io
#import fitz
from deskew import determine_skew
from skimage.transform import rotate

# Support Functions

In [16]:
# code here

## NLP

# Define the Corpus

### Apply Confidence Level Threshold

In [17]:
#path to the tesseract program on my laptop
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe" 

#***STAGE 2***
#this additional confidence check is to counter the massive char counts 
def get_confident_text(img, min_confidence=60): #define function to extract text fromimage when confidence >60%
    data = pytesseract.image_to_data(img, config='--psm 6', # use tesseract to get data about image and treat it as a single block
                                     output_type=pytesseract.Output.DATAFRAME) # return the OCR output as a DF
    # Keep only words with confidence above threshold
    data = data[(data['conf'] >= min_confidence) & (data['text'].str.strip() != '')] # only keep the data above the confidence threshold set
    return ' '.join(data['text'].tolist()) # adds text above threshold to the string


### Pre-Processing and Text Extraction

In [18]:
#***STAGE 3***
# - Create function to extract text from JPEGs
def extract_text_from_jpg(path): #define the function to read JPG
    img = Image.open(path) #open JPEG at the path using PIL
    img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)  # using OpenCV to preprocess to improve OCR accuracy, convert to greyscale

    # - Adjust for 90 degree rotation from my scans
    img_cv = cv2.rotate(img_cv, cv2.ROTATE_90_COUNTERCLOCKWISE) # turn everything 90 degrees anti-clockwise

    # - Eliminate any other small skew [PROB OVERKILL- CONSIDER REMOVING]
    angle = determine_skew(img_cv) # calc remaining rotation angle 
    if angle is not None and abs(angle) > 0.5: # initial check to make sure any angle is above 1 degrees 
        img_cv = rotate(img_cv, angle, resize=True, # rotate the image by the angle detected 
                        preserve_range=True).astype(np.uint8) # preserve pixel values 
    # - Reduce Noise
    img_cv = cv2.medianBlur(img_cv, 3) # reduce noise by applying a blur
    
    # - Pixel Variation Handling
    local_var = img_cv.var() # determine the variance
    if local_var > 2500: # setting threshold of 2500 - apply different techniques above and below this threshold
        # where high variance, use Otsu's global thresholdbest suited to this type of image
        _, img_cv = cv2.threshold(img_cv, 0, 255, #convert grayscale to binary image using threshold where everything above goes to white
                        cv2.THRESH_BINARY + cv2.THRESH_OTSU) #Otsu technique calsc the threshold auto rather than setting it manually
    else: 
        img_cv = cv2.adaptiveThreshold(img_cv, 255, # creates binary image
                        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,# uses a weighted Guassian technique to calc thresholdforlower variance images
                        cv2.THRESH_BINARY, 15, 8) # thresholds value based on 15x15 region 

    kernel = np.ones((1,1), np.uint8) # create 1 x 1 kernel/matrix
    img_cv = cv2.morphologyEx(img_cv, cv2.MORPH_OPEN, kernel) # morphological technique to clean up bright specks
    img_cv = cv2.resize(img_cv, None, fx=2, fy=2, # increase the size 2x
                interpolation=cv2.INTER_CUBIC) # apply bicubic interpolation technique when scaling up to improve image quality

    # - Generate String
    return get_confident_text(Image.fromarray(img_cv))  # covert to PIL image for ease of processing and extract text with high conf. score


# - Create function to extract text from PDFs
def extract_text_from_pdf(pdf_path): # Define function to read a PDF
    doc = pymupdf.open(pdf_path) # Open and read PDFs 
    pages = {} #create dictionary for processed text

    # - Extract and label embedded text in PDF pages
    for page_num in range(len(doc)): # loop through the pages in the doc
        page = doc[page_num] #creates object for each page
        page_name = f"{os.path.basename(pdf_path)}_page{page_num + 1}" #create descriptive label for each page
        embedded_text = page.get_text().strip() # extracts embedded text from the PDF - not using OCR
    
    # - Where low character count, use OCR
        if len(embedded_text) > 100: # check that char count >100
            pages[page_name] = embedded_text # then save those pages
        else:
            mat = pymupdf.Matrix(3, 3) # for pages with <100 chars, scale the page 3x resolution
            pix = page.get_pixmap(matrix=mat, colorspace=pymupdf.csGRAY) # render the page as a grayscale image for processeing
            img = Image.open(io.BytesIO(pix.tobytes("png"))) # converts the image/pixmap into an object that PIL can process
            #pages[page_name] = get_confident_text(img) # use OCR on the object and store high-confidence text

            # - Adjust for 90 degree rotation from my scans here too
            img_cv = np.array(img)  # already greyscale, no conversion needed
            img_cv = cv2.rotate(img_cv, cv2.ROTATE_90_COUNTERCLOCKWISE)
            img = Image.fromarray(img_cv)

            pages[page_name] = get_confident_text(img)

    return pages # outputs the 'pages' dict. with the processed results

In [ ]:
#***STAGE 4***
# - Create function to check file format before calling other function
def extract_text(path): # define function name
    ext = os.path.splitext(path)[1].lower() # split the path name and change the file ext value to lowercase
    if ext in ['.jpg', '.jpeg', '.png']: # if that ext is one of these values
        return {os.path.basename(path): extract_text_from_jpg(path)} # filename becomes dict. key and OCT extracted text becomes dict. entry
    elif ext == '.pdf': 
        return extract_text_from_pdf(path) # extracts text from PDF - each page has separate key
    else:
        print(f"Unsupported format: {path}") # focus on PDF and JPG for this exercise. Code doesn't cover other formats with certainty 
        return {}

# - Build the corpus 
document_paths = [
    'schoolpros1.pdf',
    'schoolpros2.jpg',
    'schoolpros3.pdf',
    'schoolpros4.jpg',
    'schoolpros5.pdf',
    'schoolpros6.jpg',
    'schoolpros7.pdf',
    'schoolpros8.jpg',
    'schoolpros9.pdf',
    'schoolpros10.jpg',
    'schoolpros11.pdf',
    'schoolpros12.jpg',
    'schoolpros13.pdf',
    # might adde more but that is it for now
]

corpus_raw = {} # create a new dictioanry
for path in document_paths: # loop through the docs in the path
    corpus_raw.update(extract_text(path)) # calls the functions above and merges their output into the corpus dictionary

   
# - Filter out very low text pages
corpus_raw = {k: v for k, v in corpus_raw.items() if len(v.strip()) > 50} # filter out any images or FDF pages with <100 chars
print(f"Corpus size: {len(corpus_raw)} documents that have passed char threshold") # show how many keys (images/pages) are in the corpus after filtering
for name, text in corpus_raw.items(): # looops through the corpus entries
    print(f"  {name}: {len(text.strip())} characters") # and prints the name of each image/page plus char count


## Function for Language Processing

In [ ]:
#!python -m spacy download en_core_web_sm #RAN ONCE ON MAY 3RD, SHOULDNT NEED TO RUN AGAIN

In [ ]:
# import modules etc
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import spacy

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

nlp = spacy.load("en_core_web_sm") #load the small English NLP model from spacy
stop_words = set(stopwords.words('english')) #load nltk stopwords
stemmer = PorterStemmer() #load Porter's stemmer

# set up the function to NLP pre-proc, i.e. tokenisation, remove stopswords, stem and lemma the text
def process_document(text): #define the function for pre-proc
    tokens = word_tokenize(text) #tokenise the text breaking into words and punc marks
    tokens = [t.lower() for t in tokens if t.isalpha()] #set to lowercase and remove numbers
    tokens = [t for t in tokens if t not in stop_words] #remove stopwords from tokens
    stemmed = [stemmer.stem(t) for t in tokens] #stem the tokens (porter)
    doc_spacy = nlp(" ".join(tokens)) #rejoin the tokens for lemmatisation
    lemmatised = [token.lemma_.lower() for token in doc_spacy if token.is_alpha] #lemmatise the tokens
    
    return {
        "tokens"    : tokens, #lowercase with stopwords removed
        "stemmed"   : stemmed, #stemmed versions
        "lemmatised": lemmatised, #lemmatised verstions using spacy
        "processed_text": " ".join(lemmatised) #single string of lemmatised words
    }

#Run the process_document function on the docs in the corpus
corpus_processed = {} #create a new dictionary which will hold the processed data
for doc_name, raw_text in corpus_raw.items(): #loop through each doc in the corpus
    print(f"Processing: {doc_name}") #display name of docs processed
    corpus_processed[doc_name] = process_document(raw_text) #apply the pre-proc function and send the output to the new dictionary

print("\nAll documents processed like!") #status message

# - Build corpus for TF-IDF ***DON'T THINK I'LL NEED THIS***
#tfidf_corpus = [corpus_processed[doc]["processed_text"] for doc in corpus_processed] #build list of processed text for each doc, i.e. list with 3 parts
#doc_names    = list(corpus_processed.keys()) #save list of doc names / keys

#print(f"\nCorpus ready for TF-IDF: {len(tfidf_corpus)} documents") #status message

# Language Analysis

### Tokenisation

In [ ]:
for doc_name, data in corpus_processed.items(): #loop through the docs in the corpus
    print(f"\n=== {doc_name} ===") #print doc name
    print(f"Token count: {len(data['tokens'])}") #print tokens qty
    print(f"Unique tokens: {len(set(data['tokens']))}") #print unique tokens qty
    print(f"First 5 tokens: {data['tokens'][:5]}") #as example, show first 5 tokens  

#### Token Frequency

In [ ]:
from collections import Counter #import mod to count words

for doc_name, data in corpus_processed.items(): #loop through docs in the corpus
    freq = Counter(data['tokens']) # count no. of times each token appears perdoc
    common = freq.most_common(5) # get top 5
    print(f"\nTop words in {doc_name}:") #print the output with a doc name heading
    for word, count in common: # loop through tokens and counts
        print(f"  {word}: {count}") #print the tokens and their counts

#### Unique Words per Doc

In [ ]:
doc_words = {name: set(data['tokens']) for name, data in corpus_processed.items()} # creates a new dict comprising sets of tokens

for name, words in doc_words.items(): #loop through the docs to get meaningful tokens in each doc, i.e. page
    others = set().union(*(doc_words[n] for n in doc_words if n != name)) # create 'others' to be the combo of the docs not being analysed
    unique = words - others # creates unique list by taking tokens in current doc from combo
    print(f"\nUnique meaningful words in {name}:") # print heading
    print(sorted(unique)[:5]) #prints 5 unique tokens per doc ordered alphabetically

## Stemming

In [ ]:
# - Quantify Stems per page/doc and their uniqueness

for doc_name, data in corpus_processed.items(): # loop through
    stems = data['stemmed'] # new variable for stems
    total = len(stems) # totals stems
    unique = len(set(stems)) # unique stems
    print(f"\n=== {doc_name} ===") # print each doc name on new line
    print(f"Total stems: {total}") # print heading and value 
    print(f"Unique stems: {unique}") # print heading and value
    if total == 0: #provision for images if no stems found
        print ("Stemming compression: N/A (no stems found)") #just print this msg if so
    else:
        print(f"Stemming compression: {(1 - unique/total)*100:.2f}%") #unique as % of total stems


### Common Stem Frequency

In [ ]:
for doc_name, data in corpus_processed.items(): #loop through
    freq = Counter(data['stemmed']) #quantify the stems
    common = freq.most_common(5) #top 5
    print(f"\nTop stems in {doc_name}:") #print results
    for stem, count in common:
        print(f"  {stem}: {count}")

## Lemmatisation

In [ ]:
# - Lemmas per page and their uniqueness

for doc_name, data in corpus_processed.items(): #loop through
    lemmas = data['lemmatised'] #variable lemmas
    total = len(lemmas) #total
    unique = len(set(lemmas)) #unique lemmas
    print(f"\n=== {doc_name} ===")
    print(f"Total lemmas: {total}")
    print(f"Unique lemmas: {unique}")
    if total == 0:
        print ("Lemmatisation compression: N/A (no lemmas found)")
    else:
        print(f"Lemmatisation compression: {(1 - unique/total)*100:.2f}%") #unique as % of total lemmas


### Common Lemmas Frequency

In [ ]:
all_lemmas = [] # create empty list to hold all lemmas across all documents
for doc_name, data in corpus_processed.items(): # loop through
    all_lemmas.extend(data['lemmatised']) # add each document's lemmas to the combined list

freq = Counter(all_lemmas) # count all lemmas across the entire corpus
common = freq.most_common(10) # top 10
print("Top 10 lemmas in the Prospectus:")
for lemma, count in common:
    print(f"  {lemma}: {count}")

### Lemmas Vs Stems 

In [ ]:
differences = [] #a new list for the differences b/w stems and lemmas

for doc_name, data in corpus_processed.items(): #loop through
    for token, lemma, stem in zip(data['tokens'], data['lemmatised'], data['stemmed']):
        if lemma != stem: #where lemmas and stems don't match
            differences.append((token, lemma, stem)) #add them to differences list

print(f"{'TOKEN':20} {'LEMMA':20} {'STEM':20}") #print spaced header
print("-" * 60)

for token, lemma, stem in differences[:10]: #first 10 results
    print(f"{token:20} {lemma:20} {stem:20}") #print spaced results


## TF-IDF

In [ ]:
# Importing additional libraries (maybe move up)
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

# - Create corpus string required for vectoriser
doc_names = list(corpus_raw.keys()) # convert the keys from corpus raw into list
corpus = list(corpus_raw.values()) # convert the values from corpus raw into a list (of strings)

# - Create vectoriser
vectorizer = TfidfVectorizer( # create vectoriser to turn text to numbers
    max_features=10, # use only 20 most prominent terms
    ngram_range=(1, 2) # one and two word features allowed
)
tfidf_matrix = vectorizer.fit_transform(corpus) # creates matrix repesentative of corpus docs and features

# - Create and Print dataFrame with values
feature_names = vectorizer.get_feature_names_out() # captures the features selected by vectoriser in new variable
tfidf_df = pd.DataFrame( # assign DF class to tfidf_df var
    tfidf_matrix.toarray(), # converts the matrix into array
    columns=feature_names, # column labels set to feature names
    index=doc_names        # row labels set to doc names
)
print("=== TF-IDF SCORES ===") # heading to print
print(tfidf_df.round(3)) #print the DF with results rounded to 3 dec places

# - Identify Top 5 Terms 
print("\n=== TOP TERMS PER DOCUMENT ===") # heading to print
for doc_name in doc_names: # loop throught the docs
    top = tfidf_df.loc[doc_name].sort_values(ascending=False).head(5) # sorts features by TF-IDF score per doc/row - get top 5
    print(f"\n{doc_name}:") #print doc name
    print(top.round(3)) # print result rounded to 3 places

# - Visualise Results 
fig, axes = plt.subplots( # create the fig for graphing
    len(doc_names), # 1 row per doc
    1, # 1 column (stack)
    figsize=(12, 4 * len(doc_names)) #set dimensions
)

for ax, doc_name in zip(axes, doc_names): # pairs axes and docs
    top = tfidf_df.loc[doc_name].sort_values(ascending=False).head(10) # pick top 10 per doc 
    top.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black') #set colours on graph
    ax.set_title(f'Top TF-IDF Terms — {doc_name}', fontsize=12) # set title and font size
    ax.set_xlabel('Terms', fontsize=10) # X axis label and font size 
    ax.set_ylabel('TF-IDF Score', fontsize=10) # y axis label and font size
    ax.tick_params(axis='x', rotation=45) # rotate labels to avoid overlap on display

plt.tight_layout() # auto adjust
plt.show() # display graphs

## Named Entity Recognition

In [ ]:
# - Run NER on all docs
all_entities = [] # new empty list for NER

for doc_name, text in corpus_raw.items(): #loop through docs in corpus raw
    doc = nlp(text) # create doc object using spaCy
    for ent in doc.ents: # loops through the entites found
        all_entities.append({ # new dict. for entity info
            'document':  doc_name, # add name of doc with entity
            'entity':    ent.text.strip(), # add name of entity
            'label':     ent.label_, # entity type label
            'description': spacy.explain(ent.label_) # label description
        })

# - Build new dataframe and summarise
entities_df = pd.DataFrame(all_entities) # convert the dict. for entities into a DF
print(f"Total entities found: {len(entities_df)}") #print heading and qty of entities
print(entities_df.head(20)) # print first 20 entities detected in corpus
print("\n=== ENTITIES BY TYPE ===") # print heading
print(entities_df.groupby(['label', 'description']) # group/print qty of each entity type
                 .size() # count
                 .reset_index(name='count') # new DF column for the count
                 .sort_values('count', ascending=False)) # sort highest to lowest
print("\n=== MOST FREQUENT ENTITIES ===") # print heading
print(entities_df.groupby(['entity', 'label']) # now group/print by frequency of entity occurence (not type)
                 .size() # count
                 .reset_index(name='count') # new DF column for the count
                 .sort_values('count', ascending=False) # sort highest to lowest
                 .head(20)) # top 20

# - Visualise Results 
entity_counts = entities_df.groupby('label').size().sort_values(ascending=False) # count and sort entity type high to low

plt.figure(figsize=(12, 6)) # set dimensions
entity_counts.plot(kind='bar', color='steelblue', edgecolor='black') # colours on graphs
plt.title('Entity Types Across Corpus', fontsize=14) # title and font size
plt.xlabel('Entity Type', fontsize=12) # X axis label and font
plt.ylabel('Count', fontsize=12) # y axis label and font
plt.xticks(rotation=45, ha='right') # rotate labels 45 degrees
plt.tight_layout() # auto adjust
plt.show() # and display

# Vision

## Sub Heading 1

In [ ]:
# code here...

# Multi-modal

## Sub Heading 1

In [ ]:
# code here

# Final Output

In [ ]:
# code